# Ethiopian coffee advertisement (Wan2.1-1.3B, Colab T4)
Runtime > T4 GPU. Four scenes -> Amharic titles -> Amharic narration -> final MP4.

In [ ]:
import os, glob
from google.colab import userdata
REPO = "/content/repo"
if not os.path.exists(REPO):
    tok = userdata.get("GH_TOKEN").strip()
    assert tok.startswith(("ghp_", "github_pat_")), "GH_TOKEN must be a GitHub token"
    !git clone https://{tok}@github.com/simachewalamneh/video_gen.git {REPO}
WORK = os.path.dirname(glob.glob(f"{REPO}/**/coffee_ad/gen_scenes.py", recursive=True)[0])
%cd {WORK}

In [ ]:
!apt-get -q install -y fonts-noto-core > /dev/null
!pip -q install -r requirements.txt
!pip -q uninstall -y torchao
!pip -q install uroman
!sed -i 's/from diffusers.utils import get_peft_model_state_dict/from peft.utils import get_peft_model_state_dict/' train_lora.py

**Stage 1 - generate the 4 scenes** (GPU, about 5 to 10 minutes). Set `LORA = None` for pretrained Wan only, or the adapter folder from Task 2. If you run out of memory, lower `--height 448 --width 256 --frames 25`.

In [ ]:
LORA = None   # e.g. "outputs/lora_r16"
extra = f"--lora_dir {LORA}" if LORA else ""
!python coffee_ad/gen_scenes.py --out scenes {extra}

In [ ]:
from IPython.display import Video, display
for i in range(1, 5): display(Video(f"scenes/scene{i}.mp4", embed=True, width=216))

Redo a weak scene with another seed: `!python coffee_ad/gen_scenes.py --scenes 3 --seed 7`

**Stage 2 - assemble** slow motion, dissolves and exact Amharic titles.

In [ ]:
!python coffee_ad/assemble.py --scenes scenes --out ad_silent.mp4

**Stage 3 - Amharic narration** (MMS-TTS, non-commercial licence) and final MP4.

In [ ]:
!python coffee_ad/voiceover.py --video ad_silent.mp4 --out ethiopian_coffee_ad.mp4

In [ ]:
from IPython.display import Video, display
display(Video("ethiopian_coffee_ad.mp4", embed=True, width=270))

In [ ]:
!zip -r /content/coffee_ad_results.zip scenes ad_silent.mp4 ethiopian_coffee_ad.mp4 voice.wav
from google.colab import files
files.download("/content/coffee_ad_results.zip")